# SmartBite v3 — Colab Dataset Audit
Run this notebook in Google Colab. It downloads datasets directly into the Colab runtime, audits source coverage, and saves reports to Google Drive. Nothing is downloaded to your Windows PC.

**Before running:** in Colab, add `KAGGLE_USERNAME` and `KAGGLE_KEY` under the Secrets (key icon) panel and enable notebook access. Kaggle account/API access and dataset permissions are required.

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')

import os, subprocess, sys, json, glob, shutil, pathlib, textwrap, yaml
REPO = '/content/cv_model'
DATA_HOME = '/content/smartbite_datasets'
REPORT_HOME = '/content/drive/MyDrive/SmartBite/v3_audit'
os.makedirs(DATA_HOME, exist_ok=True)
os.makedirs(REPORT_HOME, exist_ok=True)

def run(cmd, cwd=None):
    print(' $', cmd)
    subprocess.run(cmd, shell=True, check=True, cwd=cwd)

if not os.path.isdir(REPO):
    run('git clone --branch taxonomy-v3-audit --depth 1 https://github.com/ashraff-56/cv_model.git ' + REPO)
else:
    run('git fetch origin && git checkout taxonomy-v3-audit && git pull --ff-only origin taxonomy-v3-audit', cwd=REPO)

username = userdata.get('KAGGLE_USERNAME')
key = userdata.get('KAGGLE_KEY')
if not username or not key:
    raise RuntimeError('Add KAGGLE_USERNAME and KAGGLE_KEY in Colab Secrets, then rerun this cell.')
os.environ['KAGGLE_USERNAME'] = username
os.environ['KAGGLE_KEY'] = key
run('pip -q install kaggle pyyaml')
print('Repository and Kaggle authentication configured.')

## 1. Download configured datasets into Colab

This downloads the public Kaggle sources listed in `configs/data_v3.yaml` and Fruits-360. It may take a while and use substantial temporary disk space. A source that fails is recorded and the audit continues. Do not interrupt the download cell.

In [ ]:
from pathlib import Path
cfg_path = os.path.join(REPO, 'configs/data_v3.yaml')
with open(cfg_path, encoding='utf-8') as f:
    cfg = yaml.safe_load(f)

def find_marker(root, marker, max_depth=7):
    root = Path(root)
    target_parts = Path(marker).parts
    for current, dirs, files in os.walk(root):
        p = Path(current)
        try:
            rel_depth = len(p.relative_to(root).parts)
        except ValueError:
            continue
        if rel_depth > max_depth:
            dirs[:] = []
            continue
        if (p / marker).exists():
            return p
        # marker can itself be a folder path nested below current; check progressively
    return None

download_log = []
for ds_id, spec in cfg['datasets'].items():
    ref = spec.get('kaggle_ref')
    if not ref:
        download_log.append({'dataset': ds_id, 'status': 'SKIPPED_NO_KAGGLE_REF', 'ref': ''})
        continue
    dest = os.path.join(DATA_HOME, ds_id)
    os.makedirs(dest, exist_ok=True)
    marker = spec['marker']
    # Reuse a completed download if its marker is already present.
    root = find_marker(dest, marker)
    if root:
        download_log.append({'dataset': ds_id, 'status': 'REUSED', 'ref': ref, 'root': str(root)})
        spec['local'] = str(root)
        continue
    try:
        run(f'kaggle datasets download -d {ref} -p "{dest}" --unzip', cwd=REPO)
        root = find_marker(dest, marker)
        if root:
            spec['local'] = str(root)
            download_log.append({'dataset': ds_id, 'status': 'DOWNLOADED', 'ref': ref, 'root': str(root)})
        else:
            download_log.append({'dataset': ds_id, 'status': 'MARKER_NOT_FOUND', 'ref': ref, 'expected_marker': marker})
    except Exception as e:
        download_log.append({'dataset': ds_id, 'status': 'DOWNLOAD_FAILED', 'ref': ref, 'error': str(e)[:500]})
    print(download_log[-1])

# Fruits-360 repositories: sparse-checkout Training only; these are supplementary isolated-object photos.
fruit_root = Path(DATA_HOME) / 'fruits360'
fruit_root.mkdir(parents=True, exist_ok=True)
fruit_sources = [
    ('https://github.com/fruits-360/fruits-360-original-size.git', fruit_root / 'fruits-360_original-size'),
    ('https://github.com/fruits-360/fruits-360-100x100.git', fruit_root / 'fruits-360_100x100'),
]
for url, dest in fruit_sources:
    if not (dest / 'Training').exists():
        try:
            if not (dest / '.git').exists():
                run(f'git clone --depth 1 --filter=blob:none --sparse {url} "{dest}"')
            run('git sparse-checkout set Training', cwd=str(dest))
            download_log.append({'dataset': dest.name, 'status': 'DOWNLOADED_OR_PRESENT', 'root': str(dest)})
        except Exception as e:
            download_log.append({'dataset': dest.name, 'status': 'DOWNLOAD_FAILED', 'error': str(e)[:500]})
    else:
        download_log.append({'dataset': dest.name, 'status': 'REUSED', 'root': str(dest)})

# Patch only the runtime config; tracked repository config remains unchanged.
cfg['datasets']['fruits360']['local'] = str(fruit_root)

# Optional: if your private Khana subset is on Kaggle, enter its exact owner/dataset slug here.
KHANA_KAGGLE_REF = ''  # Example format: 'your-kaggle-username/smartbite-khana-subset'
if KHANA_KAGGLE_REF.strip():
    dest = os.path.join(DATA_HOME, 'khana')
    os.makedirs(dest, exist_ok=True)
    try:
        run(f'kaggle datasets download -d {KHANA_KAGGLE_REF.strip()} -p "{dest}" --unzip', cwd=REPO)
        marker = cfg['datasets']['khana']['marker']
        root = find_marker(dest, marker)
        if root:
            cfg['datasets']['khana']['local'] = str(root)
            download_log.append({'dataset':'khana','status':'DOWNLOADED','ref':KHANA_KAGGLE_REF,'root':str(root)})
        else:
            download_log.append({'dataset':'khana','status':'MARKER_NOT_FOUND','ref':KHANA_KAGGLE_REF,'expected_marker':marker})
    except Exception as e:
        download_log.append({'dataset':'khana','status':'DOWNLOAD_FAILED','ref':KHANA_KAGGLE_REF,'error':str(e)[:500]})

runtime_cfg = os.path.join(REPO, 'configs/data_v3_colab.yaml')
with open(runtime_cfg, 'w', encoding='utf-8') as f:
    yaml.safe_dump(cfg, f, sort_keys=False, allow_unicode=True)
with open(os.path.join(REPORT_HOME, 'download_log.json'), 'w') as f:
    json.dump(download_log, f, indent=2)
print('\nDOWNLOAD SUMMARY')
for row in download_log: print(row)
print('\nRuntime config:', runtime_cfg)

## 2. Run the raw source audit

This only counts files and checks configured class-to-folder mappings. It does not train a model or claim that images are valid/unique.

In [ ]:
os.chdir(REPO)
sys.path.insert(0, REPO)
run(f'python tools/audit_v3_sources.py --profile local --config {runtime_cfg} --out-dir reports/data_v3_colab')

for name in ['source_audit.csv', 'source_audit.md']:
    src = os.path.join(REPO, 'reports/data_v3_colab', name)
    if os.path.exists(src):
        shutil.copy2(src, os.path.join(REPORT_HOME, name))
print('Reports saved to:', REPORT_HOME)
print(open(os.path.join(REPO, 'reports/data_v3_colab/source_audit.md'), encoding='utf-8').read())

## 3. Package results for review

When all previous cells finish, download the ZIP from the Colab file browser or use the Drive folder. Upload `smartbite_v3_audit_results.zip` in this ChatGPT conversation. I will inspect the exact class counts, source failures and gaps before we build the visual QA and manifest stages.

In [ ]:
import zipfile
zip_path = os.path.join(REPORT_HOME, 'smartbite_v3_audit_results.zip')
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    for name in ['source_audit.csv', 'source_audit.md', 'download_log.json']:
        p = os.path.join(REPORT_HOME, name)
        if os.path.exists(p): z.write(p, arcname=name)
print('RESULT ZIP:', zip_path)
print('Drive folder:', REPORT_HOME)